In [ ]:
pip install transformers torch scikit-learn umap-learn matplotlib seaborn

In [ ]:
import torch
from transformers import BertTokenizer, BertModel
import umap.umap_ as umap
import plotly.express as px
import pandas as pd
import numpy as np

Caricamento del modello `bert-base-uncased`. Fondamentale è impostare `output_hidden_states=True` perché, di default, BERT restituisce solo l'output dell'ultimo strato.

In [ ]:
#Inizializzazione BERT, il tokenizer trasforma il testo in numeri per BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# Caricamento del modello chiedendo esplicitamente di restituire gli stati nascosti di tutti i layer, quindi tutto il ragionamento
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

Creazione di un dizionario di frasi organizzato per "target words". Il corpus è strutturato in modo da testare le capacità del modello su diversi livelli di complessità semantica: dalla monosemia (es. "cat"), all'omonimia netta (es. "bass"), fino a polisemia complessa e continuum semantici (es. "run", "paper").

In [ ]:
corpus = {
    # Bassa Complessità (Monosemia / Bassa Entropia)
    "cat": [
        "My cat loves to sleep on the sofa.",
        "The stray cat caught a mouse.",
        "Don't forget to feed the cat.",
        "The black cat crossed the street.",
        "She adopted a rescue cat from the shelter."
    ],
    "school": [ # H(w) = 0.29
        "The children are learning math at school.",
        "High school students are preparing for exams.",
        "The school board approved the new budget.",
        "He walked his daughter to school.",
        "A large school of fish swam past the diver."
    ],

    #Media Complessità (Omonimia e Polisemia Regolare)
    "bass": [ # Omonimia netta
        "He plays the bass guitar in a rock band.",
        "The bass was turned up so loud the speakers shook.",
        "We caught a large sea bass on our fishing trip.",
        "Freshwater bass is delicious when grilled.",
        "She has a deep bass voice."
    ],
    "chicken": [ # Pattern produttivo Animale-Carne
        "The farmer feeds the chicken every morning.",
        "A fox sneaked in and stole a chicken.",
        "We are having roast chicken for dinner.",
        "This chicken soup is very comforting.",
        "Don't be a chicken, just jump into the water!"
    ],
    "book": [ # Dot objects (Fisico vs Contenuto)
        "The heavy book fell off the shelf.",
        "She spilled coffee on the cover of the book.",
        "I read a fascinating book about history.",
        "He is writing a new book.",
        "Did you book the flights for our holiday?"
    ],

    #Alta Complessità (Polisemia Genuina e Continuum)
    "bank": [ # H(w) = 0.99
        "I deposited my money in the bank today.",
        "The central bank raised interest rates.",
        "She sat on the river bank and watched the water.",
        "The boat was moored on the muddy bank.",
        "The plane performed a steep bank to the left."
    ],
    "paper": [ # H(w) = 1.30 (Continuum semantico)
        "I need a blank sheet of paper to draw on.",
        "The paper is made from recycled materials.",
        "She published a groundbreaking scientific paper.",
        "I read about the scandal in the morning paper.",
        "Please sign this piece of paper."
    ],
    "head": [ # Catene radiali (struttura cognitiva)
        "He wore a hat on his head.",
        "Hit the nail on the head.",
        "She sits at the head of the table.",
        "He was appointed head of the department.",
        "The beer had a thick white head."
    ],
    "match": [
        "We watched a thrilling football match on TV.",
        "She won the tennis match in straight sets.",
        "He struck a match to light the campfire.",
        "Do you have a match for this candle?",
        "Your socks don't match your shoes."
    ],
    "run": [ # H(w) = 2.15 (Complessità estrema)
        "I go for a run in the park every morning.",
        "She knows how to run a successful business.",
        "The river will run dry if it doesn't rain soon.",
        "You have a run in your stockings.",
        "The program will run automatically on startup."
    ]
}

Definizione di una funzione per calcolare il vettore semantico della nostra parola target all'interno di una specifica frase. Utilizziamo il **penultimo strato (second-to-last layer)** di BERT. L'ultimo strato è spesso sbilanciato verso il task di pre-training originale, mentre il penultimo conserva un'ottima generalizzazione semantica.

In [ ]:
def get_embedding(sentence, target):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    # Disabilitazione del calcolo del gradiente per risparmiare memoria e velocizzare l'elaborazione, visto che non lo addestriamo ma leggiamo solo i risultati
    with torch.no_grad():
        outputs = model(**inputs)
        hidden_states = outputs.hidden_states[-2]# Usiamo il penultimo layer per una migliore resa semantica, perchè l'ultimo potrebbe esseere troppo specifico

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    try:
      # Troviamo l'indice del token corrispondente alla nostra parola target
        target_idx = tokens.index(target.lower())
        return hidden_states[0][target_idx].numpy()
    except:
        return None

iterazione su tutto il corpus per estrarre e immagazzinare gli embedding di ogni parola target in ogni contesto. Conversione successiva in un DataFrame.

In [ ]:
#Estraiamo gli embeddings
data = []
# Iterazione su ogni parola target e le relative frasi
for word, sentences in corpus.items():
    for sent in sentences:
        emb = get_embedding(sent, word)
        if emb is not None:
          # Salvataggio parola, frase di contesto e il vettore calcolato
            data.append({'word': word, 'sentence': sent, 'embedding': emb})

df_data = pd.DataFrame(data)
embeddings_matrix = np.stack(df_data['embedding'].values)

Gli embedding di BERT si trovano in uno spazio a 768 dimensioni, impossibile da visualizzare direttamente. Utilizziamo UMAP (Uniform Manifold Approximation and Projection) per proiettare questi vettori in uno spazio 2D. Rispetto a PCA o t-SNE, UMAP aiuta nel preservare sia la struttura locale che quella globale dei dati, evidenziando bene i cluster semantici.

In [ ]:
# utiliziamo UMAP per il bilanciamento
# n_neighbors=5 favorisce l'attenzione sulla struttura locale (utile per dataset piccoli).
# min_dist=0.3 controlla quanto i punti possono essere compattati nei cluster dando della distanza di separazione.
reducer = umap.UMAP(n_neighbors=5, min_dist=0.3, random_state=42)
projections = reducer.fit_transform(embeddings_matrix)
# Aggiungiamo le nuove coordinate bidimensionali al nostro DataFrame
df_data['x'], df_data['y'] = projections[:, 0], projections[:, 1]

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
fig = px.scatter(
    df_data, x='x', y='y', color='word',
    hover_data={'sentence': True, 'x': False, 'y': False},
    title="Esplorazione Interattiva della Polisemia (BERT + UMAP)",
    labels={'word': 'Parola Target'}
)

fig.update_traces(marker=dict(size=12, opacity=0.8, line=dict(width=1, color='white')))
fig.show()

### Analisi dello Spazio Semantico (BERT + UMA

*   **Monosemia (Bassa Entropia):**
    *   **`cat`** (blu) e **`chicken`** (viola) formano cluster molto densi e ravvicinati.
    *   *Interpretazione:* Il significato è stabile; la rappresentazione vettoriale subisce minime variazioni contestuali.

*   **Omonimia Netta (Cluster Multipli e Outlier):**
    *   **`school`** (rosso): Mostra un cluster principale (istruzione) e un chiaro **outlier** distante (relativo all'accezione "school of fish", banco di pesci).
    *   **`bass`** (verde): I punti formano sub-cluster separati, distinguendo nettamente i domini (strumento musicale vs. pesce).

*   **Polisemia e Continuum Semantico (Alta Entropia):**
    *   **`run`** (giallo): Presenta la **massima dispersione** spaziale. Crea un vero e proprio continuum semantico che copre sfumature molto diverse (correre, gestire un'azienda, eseguire codice).
    *   **`bank`** (azzurro): Punti molto distanziati che riflettono la classica polarizzazione tra istituto finanziario e riva del fiume.
    *   **`paper`** (rosa) e **`head`** (verde chiaro): Mostrano una varianza moderata, tipica degli slittamenti di significato cognitivo (es. materiale fisico vs. articolo scientifico).

**Conclusione:** A differenza dei modelli statici in cui i significati collassano in un unico vettore medio, la pipeline BERT + UMAP modella con precisione la topologia dinamica della polisemia.